In [2]:
# Cell 1: Install lean dependencies (Zero API keys needed)
!pip install -q pypdf sentence-transformers faiss-gpu-cu12 transformers torch

In [3]:
# Cell 2: Deterministic Regex-based Clause Splitter
import re
from typing import List, Dict

SAMPLE_LEGAL_CONTRACT = """
AGREEMENT FOR SERVICES
This Agreement is entered into by and between Alpha Corp ("Client") and Beta LLC ("Provider").

1. SCOPE AND SERVICES
Provider agrees to deliver backend engineering and data pipeline services as detailed in Exhibit A.
All deliverables shall be provided within forty-five (45) business days of execution.

2. COMPENSATION AND PAYMENT
Client shall pay Provider a fixed fee of $25,000 upon successful completion of the milestones.
Invoices not paid within thirty (30) calendar days will accrue late fees at 1.5% per month.

3. LIMITATION OF LIABILITY
EXCEPT FOR GROSS NEGLIGENCE OR WILLFUL MISCONDUCT, NEITHER PARTY SHALL BE LIABLE FOR
INDIRECT, INCIDENTAL, OR CONSEQUENTIAL DAMAGES. PROVIDER'S TOTAL AGGREGATE LIABILITY
ARISING OUT OF OR RELATED TO THIS AGREEMENT SHALL BE STRICTLY LIMITED TO THE TOTAL FEES
PAID BY CLIENT IN THE PRECEDING TWELVE (12) MONTH PERIOD.

4. TERMINATION FOR CONVENIENCE
Either party may terminate this Agreement without cause upon giving thirty (30) days prior
written notice to the other party. In the event of such termination, Client shall pay for
all verifiable work completed up to the termination date.

5. GOVERNING LAW AND JURISDICTION
This Agreement shall be construed, interpreted, and governed in accordance with the laws
of the State of Delaware, without regard to its conflicts of law provisions.
"""

def extract_clauses(text: str, doc_name: str = "doc_01") -> List[Dict]:
    """
    Splits text on numbered clause headings (e.g., '1. SCOPE', 'Clause 2:', 'Section 3.1').
    Maintains header-to-body integrity to prevent lost context.
    """
    pattern = r'(?=(\n\s*\d+[\.\)]\s+[A-Z\s]{3,}))'
    raw_sections = re.split(pattern, text)

    chunks = []
    chunk_id = 0

    for section in raw_sections:
        clean = section.strip()
        if len(clean) > 40: # Discard short whitespace artifacts
            # Extract the header line
            lines = clean.split('\n')
            title = lines[0].strip()
            body = "\n".join(lines[1:]).strip() if len(lines) > 1 else lines[0]

            chunks.append({
                "chunk_id": chunk_id,
                "doc_name": doc_name,
                "clause_title": title,
                "text": clean
            })
            chunk_id += 1

    return chunks

parsed_chunks = extract_clauses(SAMPLE_LEGAL_CONTRACT)
print(f"Extracted {len(parsed_chunks)} structured clauses.")
for c in parsed_chunks:
    print(f" -> [{c['chunk_id']}] {c['clause_title']}")

Extracted 8 structured clauses.
 -> [0] AGREEMENT FOR SERVICES
 -> [1] 1. SCOPE AND SERVICES
 -> [2] 2. COMPENSATION AND PAYMENT
 -> [3] 3. LIMITATION OF LIABILITY
 -> [4] 3. LIMITATION OF LIABILITY
 -> [5] 3. LIMITATION OF LIABILITY
 -> [6] 4. TERMINATION FOR CONVENIENCE
 -> [7] 5. GOVERNING LAW AND JURISDICTION


In [4]:
# Cell 3: Offline Index Generation & Serialization
import json
import numpy as np
import faiss
from sentence_transformers import SentenceTransformer

# 1. Load small footprint open-weight embedding model (~130MB RAM)
embedding_model = SentenceTransformer('BAAI/bge-small-en-v1.5')

# 2. Extract texts and generate normalized embeddings for cosine similarity
texts = [c['text'] for c in parsed_chunks]
embeddings = embedding_model.encode(texts, normalize_embeddings=True, show_progress_bar=True)
embeddings = np.array(embeddings, dtype=np.float32)

# 3. Build FAISS Inner Product (Cosine) Index
dimension = embeddings.shape[1]
index = faiss.IndexFlatIP(dimension)
index.add(embeddings)

# 4. Serialize to disk (ready for export to VS Code)
faiss.write_index(index, "contract_index.faiss")
with open("chunk_metadata.json", "w") as f:
    json.dump(parsed_chunks, f, indent=2)

print(f"✅ FAISS index saved ({index.ntotal} vectors). Metadata saved to chunk_metadata.json.")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

✅ FAISS index saved (8 vectors). Metadata saved to chunk_metadata.json.


In [5]:
# Cell 2: Download official CUAD v1 text dataset directly
import json
import os
import urllib.request
import zipfile

cuad_zip_url = "https://github.com/TheAtticusProject/cuad/raw/main/data.zip"
zip_path = "data.zip"
json_path = "CUADv1.json"

if not os.path.exists(json_path):
    print("Downloading CUAD dataset archive (~17MB)...")
    urllib.request.urlretrieve(cuad_zip_url, zip_path)
    print("Extracting CUADv1.json...")
    with zipfile.ZipFile(zip_path, "r") as zip_ref:
        zip_ref.extractall(".")
    print("Extraction complete.")
else:
    print("CUADv1.json already present.")

# Load the SQuAD-formatted CUAD dataset
with open(json_path, "r", encoding="utf-8") as f:
    cuad_data = json.load(f)

# Extract the very first contract context
first_doc = cuad_data["data"][0]
doc_title = first_doc.get("title", "Contract_0")
raw_text = first_doc["paragraphs"][0]["context"]

print("\n" + "="*50)
print(f"Loaded Document Title: {doc_title}")
print(f"Total Characters: {len(raw_text)}")
print(f"Total Words: {len(raw_text.split())}")
print(f"Total Paragraphs/Clauses in JSON: {len(first_doc['paragraphs'])}")
print("="*50)
print("\n--- First 600 Characters of Contract ---\n")
print(raw_text[:600])
print("\n" + "="*50)

Extracting CUADv1.json...
Extraction complete.

Loaded Document Title: LIMEENERGYCO_09_09_1999-EX-10-DISTRIBUTOR AGREEMENT
Total Characters: 54290
Total Words: 5998
Total Paragraphs/Clauses in JSON: 1

--- First 600 Characters of Contract ---

EXHIBIT 10.6

                              DISTRIBUTOR AGREEMENT

         THIS  DISTRIBUTOR  AGREEMENT (the  "Agreement")  is made by and between Electric City Corp.,  a Delaware  corporation  ("Company")  and Electric City of Illinois LLC ("Distributor") this 7th day of September, 1999.

                                    RECITALS

         A. The  Company's  Business.  The Company is  presently  engaged in the business  of selling an energy  efficiency  device,  which is  referred to as an "Energy  Saver"  which may be improved  or  otherwise  changed  from its present composition (the "P



In [6]:
# Cell 3: Structure-Aware Legal Clause Parser
import re
from typing import List, Dict, Any

class StructureAwareLegalParser:
    def __init__(self, min_clause_length: int = 40):
        self.min_clause_length = min_clause_length
        # Pattern catches: "SECTION 1", "Section 1.1", "ARTICLE IV", "1.1 Title", or lettered recitals "A. Title"
        self.section_pattern = re.compile(
            r"(?:\n\s*)"
            r"(?:"
            r"(?:ARTICLE|SECTION|CLAUSE)\s+[0-9IVXLCDM]+(?:\.[0-9]+)*[^\n]*|"
            r"(?:[0-9]{1,2}\.[0-9]{1,2}(?:\.[0-9]+)*)\s+[^\n]+|"
            r"(?:[A-Z]\.\s+[^\n]+)|"
            r"(?:RECITALS|DEFINITIONS|MISCELLANEOUS|TERMINATION|CONFIDENTIALITY|INDEMNIFICATION)"
            r")",
            re.IGNORECASE
        )

    def parse(self, text: str, doc_id: str = "doc_0") -> List[Dict[str, Any]]:
        matches = list(self.section_pattern.finditer(text))
        clauses = []

        if not matches:
            # Fallback if no strict headers found
            return [{
                "clause_id": f"{doc_id}_clause_0",
                "header": "FULL_DOCUMENT",
                "text": text.strip(),
                "start_char": 0,
                "end_char": len(text)
            }]

        # Preamble / Text before the first matched header
        if matches[0].start() > 0:
            preamble_text = text[:matches[0].start()].strip()
            if len(preamble_text) >= self.min_clause_length:
                clauses.append({
                    "clause_id": f"{doc_id}_clause_0",
                    "header": "PREAMBLE",
                    "text": preamble_text,
                    "start_char": 0,
                    "end_char": matches[0].start()
                })

        for i, match in enumerate(matches):
            start_pos = match.start()
            end_pos = matches[i + 1].start() if i + 1 < len(matches) else len(text)

            clause_raw = text[start_pos:end_pos].strip()
            header_line = match.group(0).strip()

            # Clean up the body text
            body_text = clause_raw[len(header_line):].strip()
            full_clause_text = f"{header_line}\n{body_text}".strip()

            if len(full_clause_text) >= self.min_clause_length:
                clauses.append({
                    "clause_id": f"{doc_id}_clause_{len(clauses)}",
                    "header": header_line,
                    "text": full_clause_text,
                    "start_char": start_pos,
                    "end_char": end_pos
                })

        return clauses

# Execute parsing on our sample contract
parser = StructureAwareLegalParser()
parsed_clauses = parser.parse(raw_text, doc_id="LIME_1999_EX10")

print("="*60)
print(f"Total Structural Clauses Extracted: {len(parsed_clauses)}")
print("="*60)

# Inspect first 3 parsed clauses
for c in parsed_clauses[:3]:
    print(f"\n[ID]: {c['clause_id']}")
    print(f"[Header]: {c['header']}")
    print(f"[Offsets]: {c['start_char']} -> {c['end_char']}")
    print(f"[Sample Content]:\n{c['text'][:220]}...")
    print("-" * 50)

Total Structural Clauses Extracted: 45

[ID]: LIME_1999_EX10_clause_0
[Header]: PREAMBLE
[Offsets]: 0 -> 290
[Sample Content]:
EXHIBIT 10.6

                              DISTRIBUTOR AGREEMENT

         THIS  DISTRIBUTOR  AGREEMENT (the  "Agreement")  is made by and between Electric City Corp.,  a Delaware  corporation  ("Company")  and Electric...
--------------------------------------------------

[ID]: LIME_1999_EX10_clause_1
[Header]: A. The  Company's  Business.  The Company is  presently  engaged in the business  of selling an energy  efficiency  device,  which is  referred to as an "Energy  Saver"  which may be improved  or  otherwise  changed  from its present composition (the "Products").  The Company may engage in the business of selling other  products  or  other  devices  other  than  the  Products,  which  will be considered  Products if Distributor  exercises its options pursuant to Section 7 hereof.
[Offsets]: 336 -> 831
[Sample Content]:
A. The  Company's  Business.  Th

In [7]:
# Cell 4: Dense Embedding & Index Serialization
import json
import time
import numpy as np
import faiss
from sentence_transformers import SentenceTransformer

# 1. Load lightweight embedding model on GPU if available
embedding_model_name = "BAAI/bge-small-en-v1.5"
print(f"Loading embedding model: {embedding_model_name}...")
embedder = SentenceTransformer(embedding_model_name)

# 2. Prepare text with BGE passage prefix for asymmetric retrieval
passage_prefix = "passage: "
clause_texts = [passage_prefix + c["text"] for c in parsed_clauses]

# 3. Generate normalized embeddings (so Inner Product == Cosine Similarity)
print(f"Generating embeddings for {len(clause_texts)} clauses...")
t0 = time.time()
embeddings = embedder.encode(
    clause_texts,
    normalize_embeddings=True,
    show_progress_bar=False,
    convert_to_numpy=True
)
t_embed = (time.time() - t0) * 1000
print(f"Embedding completed in {t_embed:.2f} ms ({t_embed / len(clause_texts):.2f} ms/clause)")

# 4. Build FAISS Index
embedding_dim = embeddings.shape[1]
index = faiss.IndexFlatIP(embedding_dim)
index.add(embeddings)
print(f"FAISS Index built. Total vectors indexed: {index.ntotal}")

# 5. Serialize Artifacts to Disk
index_file = "contract_index.faiss"
meta_file = "clauses_metadata.json"

faiss.write_index(index, index_file)
with open(meta_file, "w", encoding="utf-8") as f:
    json.dump(parsed_clauses, f, indent=2)

print(f"\n[Saved] FAISS index -> {index_file} ({round(os.path.getsize(index_file)/1024, 2)} KB)")
print(f"[Saved] Metadata JSON -> {meta_file} ({round(os.path.getsize(meta_file)/1024, 2)} KB)")

# 6. Verify Cold Start Loading from Disk
t_load_start = time.time()
loaded_index = faiss.read_index(index_file)
with open(meta_file, "r", encoding="utf-8") as f:
    loaded_meta = json.load(f)
t_load_end = (time.time() - t_load_start) * 1000

print(f"\n[Verification] Index & Metadata reloaded in {t_load_end:.2f} ms")
assert loaded_index.ntotal == len(loaded_meta) == len(parsed_clauses)
print("State validation successful: Zero data loss during disk serialization.")

Loading embedding model: BAAI/bge-small-en-v1.5...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Generating embeddings for 45 clauses...
Embedding completed in 415.84 ms (9.24 ms/clause)
FAISS Index built. Total vectors indexed: 45

[Saved] FAISS index -> contract_index.faiss (67.54 KB)
[Saved] Metadata JSON -> clauses_metadata.json (86.61 KB)

[Verification] Index & Metadata reloaded in 0.87 ms
State validation successful: Zero data loss during disk serialization.


In [8]:
# Cell 5: Query Execution & Retrieval Benchmarking
import time
import numpy as np

def retrieve_clauses(query: str, top_k: int = 3):
    t_start = time.time()

    # Format query with BGE specific retrieval prompt
    formatted_query = f"Represent this sentence for searching relevant passages: {query}"

    # 1. Encode query
    query_vec = embedder.encode(
        [formatted_query],
        normalize_embeddings=True,
        convert_to_numpy=True
    )

    # 2. Search FAISS Index (Cosine similarity via Inner Product)
    scores, indices = loaded_index.search(query_vec, top_k)
    latency_ms = (time.time() - t_start) * 1000

    # 3. Format results
    results = []
    for rank, (score, idx) in enumerate(zip(scores[0], indices[0]), start=1):
        clause_data = loaded_meta[idx]
        results.append({
            "rank": rank,
            "score": float(score),
            "clause_id": clause_data["clause_id"],
            "header": clause_data["header"],
            "text": clause_data["text"],
            "offsets": (clause_data["start_char"], clause_data["end_char"])
        })

    return results, latency_ms

# Test queries on the contract
test_queries = [
    "What are the products or devices being sold under this agreement?",
    "Under what conditions can the distributor or company terminate the contract?",
    "What are the representations and financial capabilities required?"
]

print("="*70)
print("RETRIEVAL LATENCY AND MATCH EVALUATION")
print("="*70)

for q in test_queries:
    hits, latency = retrieve_clauses(q, top_k=2)
    print(f"\nQuery: '{q}'")
    print(f"Retrieval Latency: {latency:.2f} ms")
    for hit in hits:
        print(f"  -> Rank {hit['rank']} [Score: {hit['score']:.4f}] | Clause: {hit['clause_id']}")
        print(f"     Header: {hit['header'][:80]}...")
        print(f"     Preview: {hit['text'][:120].replace(chr(10), ' ')}...\n")
    print("-" * 70)

RETRIEVAL LATENCY AND MATCH EVALUATION

Query: 'What are the products or devices being sold under this agreement?'
Retrieval Latency: 35.76 ms
  -> Rank 1 [Score: 0.7050] | Clause: LIME_1999_EX10_clause_44
     Header: 7.3      Other  Agreements.  The terms  pursuant  to which  such  other         ...
     Preview: 7.3      Other  Agreements.  The terms  pursuant  to which  such  other                   Products  or devices  shall be...

  -> Rank 2 [Score: 0.6998] | Clause: LIME_1999_EX10_clause_11
     Header: 1.7      Relationship of Parties.  The relationship between the Company         ...
     Preview: 1.7      Relationship of Parties.  The relationship between the Company                   and the Distributor Page -3-  ...

----------------------------------------------------------------------

Query: 'Under what conditions can the distributor or company terminate the contract?'
Retrieval Latency: 11.02 ms
  -> Rank 1 [Score: 0.7576] | Clause: LIME_1999_EX10_clause_12
     Heade

In [9]:
# Cell 6: NLI Hallucination & Grounding Gate
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification

nli_model_name = "cross-encoder/nli-deberta-v3-small"
print(f"Loading NLI Verification Gate: {nli_model_name}...")

device = "cuda" if torch.cuda.is_available() else "cpu"
nli_tokenizer = AutoTokenizer.from_pretrained(nli_model_name)
nli_model = AutoModelForSequenceClassification.from_pretrained(nli_model_name).to(device)
nli_model.eval()

# Label mapping for DeBERTa NLI: 0 = Contradiction, 1 = Entailment, 2 = Neutral
label_mapping = {0: "CONTRADICTION", 1: "ENTAILMENT", 2: "NEUTRAL"}

def verify_grounding(premise_text: str, generated_claim: str, entailment_threshold: float = 0.70):
    t0 = time.time()

    # Format input pair for Cross-Encoder
    inputs = nli_tokenizer(
        premise_text,
        generated_claim,
        truncation=True,
        max_length=512,
        return_tensors="pt"
    ).to(device)

    with torch.no_grad():
        logits = nli_model(**inputs).logits
        probs = torch.softmax(logits, dim=-1)[0].cpu().numpy()

    latency_ms = (time.time() - t0) * 1000

    prob_contradiction = float(probs[0])
    prob_entailment = float(probs[1])
    prob_neutral = float(probs[2])

    # Decision Gate Logic
    if prob_entailment >= entailment_threshold:
        status = "VERIFIED_SAFE"
    elif prob_contradiction > 0.40:
        status = "HALLUCINATION_CONTRADICTION"
    else:
        status = "UNGROUNDED_NEUTRAL"

    return {
        "status": status,
        "entailment_prob": prob_entailment,
        "neutral_prob": prob_neutral,
        "contradiction_prob": prob_contradiction,
        "latency_ms": latency_ms
    }

# Retrieve the actual Termination clause text from our parsed list
termination_clause = [c for c in parsed_clauses if "4.2" in c["header"]][0]["text"]

print("\n--- Source Clause (Premise) ---")
print(termination_clause[:350] + "...\n" + "="*70)

# Define test cases: True claim, subtle hallucination, and blatant contradiction
test_cases = [
    {
        "type": "Faithful Summary (True)",
        "claim": "Either party may terminate the agreement upon 30 days written notice for cause."
    },
    {
        "type": "Factual Hallucination (Wrong Number)",
        "claim": "Either party may terminate the agreement immediately upon 90 days notice."
    },
    {
        "type": "Blatant Contradiction",
        "claim": "The agreement cannot be terminated for cause under any circumstances."
    },
    {
        "type": "Ungrounded Extrapolation",
        "claim": "The distributor must pay a fine of 5000 dollars before terminating."
    }
]

print("NLI VERIFICATION BENCHMARK RESULTS")
print("="*70)

for case in test_cases:
    res = verify_grounding(termination_clause, case["claim"])
    print(f"Test Type: {case['type']}")
    print(f"Claim: \"{case['claim']}\"")
    print(f"Decision: [{res['status']}] (Latency: {res['latency_ms']:.2f} ms)")
    print(f"Probabilities -> Entailment: {res['entailment_prob']:.4f} | Contradiction: {res['contradiction_prob']:.4f} | Neutral: {res['neutral_prob']:.4f}")
    print("-" * 70)

Loading NLI Verification Gate: cross-encoder/nli-deberta-v3-small...


config.json:   0%|          | 0.00/1.05k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/1.35k [00:00<?, ?B/s]

spm.model: reconstructing file:   0%|          |  0.00B / 2.46MB            

spm.model: downloading bytes:           |  0.00B            

...

special_tokens_map.json:   0%|          | 0.00/301 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  568MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/106 [00:00<?, ?it/s]


--- Source Clause (Premise) ---
4.2      Termination  for  Cause.   Either  party  may  terminate  this                   Agreement upon 30 days
Page -8-

                  prior written  notice to the other upon the  occurrence of any                   of the following events: (A) the Distributor's failure to make                   full and  prompt  payment  to the  Company of a...
NLI VERIFICATION BENCHMARK RESULTS
Test Type: Faithful Summary (True)
Claim: "Either party may terminate the agreement upon 30 days written notice for cause."
Decision: [VERIFIED_SAFE] (Latency: 537.58 ms)
Probabilities -> Entailment: 0.9931 | Contradiction: 0.0003 | Neutral: 0.0066
----------------------------------------------------------------------
Test Type: Factual Hallucination (Wrong Number)
Claim: "Either party may terminate the agreement immediately upon 90 days notice."
Decision: [HALLUCINATION_CONTRADICTION] (Latency: 424.38 ms)
Probabilities -> Entailment: 0.0009 | Contradiction: 0.9831 | Neut

In [10]:
# Cell 7: Full End-to-End Audited Engine
import json
from typing import Dict, Any, List

def run_audited_legal_query(query: str, top_k: int = 2) -> Dict[str, Any]:
    print(f"\n=======================================================")
    print(f"QUERY: '{query}'")
    print(f"=======================================================")

    # 1. Retrieval Step
    retrieved_hits, retrieval_time = retrieve_clauses(query, top_k=top_k)
    best_match = retrieved_hits[0]

    print(f"[1. Retrieval] Top Clause: {best_match['clause_id']} | Score: {best_match['score']:.4f} | Time: {retrieval_time:.2f}ms")
    print(f"    Header: {best_match['header']}")

    # 2. Simulated Structured Output (Mimicking a local SLM output)
    # In VS Code, this will be produced via llama.cpp/vLLM with strict JSON schema
    simulated_claims = [
        "Either party can terminate with 30 days written notice upon specific default events.",
        "The contract imposes an automatic 10000 dollar penalty upon termination."
    ]

    # 3. NLI Auditing Step across all claims
    audited_results = []
    print("\n[2. Safety & Verification Audit]")

    for claim in simulated_claims:
        audit = verify_grounding(best_match["text"], claim)
        is_safe = audit["status"] == "VERIFIED_SAFE"

        audited_results.append({
            "claim": claim,
            "status": audit["status"],
            "is_safe": is_safe,
            "entailment_score": audit["entailment_prob"],
            "contradiction_score": audit["contradiction_prob"]
        })

        status_flag = "PASS" if is_safe else "FAIL"
        print(f"  [{status_flag}] Status: {audit['status']}")
        print(f"         Claim: \"{claim}\"")
        print(f"         Entailment: {audit['entailment_prob']:.4f} | Contradiction: {audit['contradiction_prob']:.4f}")

    # 4. Filter only verified claims for final response
    safe_output = [r["claim"] for r in audited_results if r["is_safe"]]
    flagged_output = [r["claim"] for r in audited_results if not r["is_safe"]]

    return {
        "query": query,
        "retrieved_clause_id": best_match["clause_id"],
        "retrieved_header": best_match["header"],
        "verified_claims": safe_output,
        "flagged_hallucinations": flagged_output,
        "audit_details": audited_results
    }

# Execute end-to-end run
e2e_result = run_audited_legal_query("Under what conditions can the contract be terminated for cause?")


QUERY: 'Under what conditions can the contract be terminated for cause?'
[1. Retrieval] Top Clause: LIME_1999_EX10_clause_24 | Score: 0.7513 | Time: 50.07ms
    Header: 4.2      Termination  for  Cause.   Either  party  may  terminate  this                   Agreement upon 30 days

[2. Safety & Verification Audit]
  [PASS] Status: VERIFIED_SAFE
         Claim: "Either party can terminate with 30 days written notice upon specific default events."
         Entailment: 0.9927 | Contradiction: 0.0002
  [FAIL] Status: UNGROUNDED_NEUTRAL
         Claim: "The contract imposes an automatic 10000 dollar penalty upon termination."
         Entailment: 0.0000 | Contradiction: 0.0033


In [11]:
# Cell 8: Bundle and download persistent artifacts
import zipfile
from google.colab import files

artifact_zip = "legal_safety_artifacts.zip"
with zipfile.ZipFile(artifact_zip, "w") as z:
    z.write("contract_index.faiss")
    z.write("clauses_metadata.json")

print(f"Artifact package created: {artifact_zip}")
files.download(artifact_zip)

Artifact package created: legal_safety_artifacts.zip


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>